# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import duckdb
import pandas as pd
import numpy as np

from bms_sa_review.synthetic_ami_creation.config import ami_config as Config

from bms_sa_review.ami_analysis.lib.ami_conformance_plot import (
    plot_assessable_vvar_window
)

from bms_sa_review.ami_analysis.lib import ami_conformance_plot as ConformancePlot
from bms_sa_review.ami_analysis.lib import ami_polarity as Polarity

import io
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 1. Connection

In [ ]:
# ---------------------------------------------------------------------------
# connect + register the views this notebook needs
# ---------------------------------------------------------------------------
# ami_meter_residential is Hive-partitioned by dt_month, 
# so it needs the glob + hive_partitioning=1 form.
# ami_site_metadata_residential is a single parquet file, no partitioning.

con = duckdb.connect()

con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_residential AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_meter_residential").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")

con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata_residential AS
    SELECT * FROM read_parquet('{Config.store_path("ami_site_metadata_residential").as_posix()}')
""")

# 2. Data shape

In [ ]:
# ---------------------------------------------------------------------------
# Get oriented on ami_meter_residential's shape/coverage WITHOUT pulling the full table into pandas
# ---------------------------------------------------------------------------

overview = con.sql("""
    SELECT
        count(*)                    AS n_rows,
        count(DISTINCT site_id)     AS n_sites,
        count(DISTINCT device_id)   AS n_devices,
        count(DISTINCT circuit_id)  AS n_circuits,
        min(t_stamp)                AS t_stamp_min,
        max(t_stamp)                AS t_stamp_max
    FROM ami_meter_residential
""").df()

print(f"ami_meter_residential: {overview.n_rows[0]:,} rows")
print(f"Distinct sites:    {overview.n_sites[0]:,}")
print(f"Distinct devices:  {overview.n_devices[0]:,}")
print(f"Distinct circuits: {overview.n_circuits[0]:,}")
print(f"t_stamp range:     {overview.t_stamp_min[0]} -> {overview.t_stamp_max[0]}")

# dtypes without materializing rows
con.sql("DESCRIBE ami_meter_residential")

# 3. Phase count

In [ ]:
# ---------------------------------------------------------------------------
# Derive load-side phase count per site
# ---------------------------------------------------------------------------
# "1-phase house load" = exactly one ac_load_net circuit_id at that site.

load_phase_count = con.sql("""
    SELECT
        site_id,
        COUNT(DISTINCT circuit_id) AS n_load_circuits,
        COUNT(DISTINCT device_id)  AS n_load_devices
    FROM ami_meter_residential
    GROUP BY site_id
""").df()

print("Distribution of n_load_circuits per site:")
print(load_phase_count['n_load_circuits'].value_counts().sort_index())

# Flag sites that are genuinely single-phase on the load side.
one_phase_load_sites = load_phase_count.loc[
    load_phase_count['n_load_circuits'] == 1, 'site_id'
]
print(f"\n1-phase-load sites: {len(one_phase_load_sites):,} / {len(load_phase_count):,}")

# 4. Cohort meta

In [ ]:
# ---------------------------------------------------------------------------
# Restrict to the cohort directly in SQL 
# site metadata is small enough to load fully
# ---------------------------------------------------------------------------

site_meta = con.sql("SELECT * FROM ami_site_metadata_residential").df()

print(f"ami_site_metadata_residential: {len(site_meta):,} rows")
print(f"Distinct site_id: {site_meta['site_id'].nunique():,}  "
      f"(1 row per site? {len(site_meta) == site_meta['site_id'].nunique()})")

cohort_meta = site_meta.loc[
    (site_meta['n_load_phases'] == 1) & (site_meta['n_pv_phases'] == 1)
].copy()

print(f"\nSites with n_load_phases==1 & n_pv_phases==1: {len(cohort_meta):,} / {len(site_meta):,}")
print(cohort_meta['in_ami_meter'].value_counts())

not_in_meter = cohort_meta.loc[~cohort_meta['in_ami_meter']]
if len(not_in_meter):
    print(f"\n{len(not_in_meter)} cohort sites are flagged NOT in ami_meter -- "
          f"these will silently drop out of the filters below.")

# Register the cohort site_id list as a DuckDB table so all queries below
# filter with a join, not a giant Python IN-list or a full-table pull.
con.register("cohort_site_ids_df", cohort_meta[['site_id']])

# 5. Cohort restrict - Random rows

In [ ]:
# Overview via SQL 
cohort_overview = con.sql("""
    SELECT
        count(*)                    AS n_rows,
        count(DISTINCT m.site_id)   AS n_sites,
        min(m.t_stamp)               AS t_stamp_min,
        max(m.t_stamp)               AS t_stamp_max
    FROM ami_meter_residential m
    JOIN cohort_site_ids_df c USING (site_id)
""").df()

print(f"\nCohort interval rows: {cohort_overview.n_rows[0]:,}")
print(f"Cohort sites present in ami_meter_residential: {cohort_overview.n_sites[0]:,} "
      f"/ {len(cohort_meta):,} in metadata")
print(f"t_stamp range: {cohort_overview.t_stamp_min[0]} -> {cohort_overview.t_stamp_max[0]}")

In [ ]:
# Per-site row counts
rows_per_site = con.sql("""
    SELECT
        m.site_id,
        count(*)                     AS n_rows,
        count(DISTINCT m.circuit_id) AS n_circuits,
        min(m.t_stamp)                AS t_stamp_min,
        max(m.t_stamp)                AS t_stamp_max
    FROM ami_meter_residential m
    JOIN cohort_site_ids_df c USING (site_id)
    GROUP BY m.site_id
""").df()

bad = rows_per_site.loc[rows_per_site['n_circuits'] != 1]
assert len(bad) == 0, (
    f"{len(bad)} cohort sites have != 1 circuit_id in ami_meter_residential -- "
    f"cohort definition or join is broken."
)
print(f"\nConfirmed: exactly 1 circuit_id per site across all {len(rows_per_site):,} cohort sites.")
print("\nRows-per-site distribution:")
print(rows_per_site['n_rows'].describe())


In [ ]:
rows_per_site['n_rows'].describe()

In [ ]:
##########################################################
### RANDOM SMALL SAMPLE TO WORK AROUND THE MEMORYERROR ###
##########################################################

SAMPLE_N = 200_000

cohort_sample = con.sql(f"""
    SELECT m.*
    FROM ami_meter_residential m
    JOIN cohort_site_ids_df c USING (site_id)
    USING SAMPLE {SAMPLE_N} ROWS
""").df()

print(f"\nCohort sample: {len(cohort_sample):,} rows, "
      f"{cohort_sample['site_id'].nunique():,} distinct sites represented")

In [ ]:
# ---------------------------------------------------------------------------
# Shape, coverage, and basic quality pass on the cohort
# ---------------------------------------------------------------------------

print("=== Shape (full cohort, via SQL aggregates) ===")
print(f"Sites: {cohort_overview.n_sites[0]:,}")
print(f"Rows:  {cohort_overview.n_rows[0]:,}")
print(f"Date range: {cohort_overview.t_stamp_min[0]} -> {cohort_overview.t_stamp_max[0]}")

print("\n=== Rows per site (full cohort, coverage) ===")
print(rows_per_site['n_rows'].describe())

print("\n=== Missingness (% null, sample of "
      f"{len(cohort_sample):,} rows) ===")
print((cohort_sample.isna().mean() * 100).round(2))

print("\n=== Basic distributions (sample) ===")
print(cohort_sample[['V', 'P_kw', 'Q_kvar', 'S_kva', 'power_factor', 'current_a']].describe())

# Voltage sanity bound, mirroring the 0<V<300 pattern
bad_voltage = cohort_sample.loc[~cohort_sample['V'].between(0, 300, inclusive='neither')]
print(f"\nSample rows with V outside (0, 300): {len(bad_voltage):,} "
      f"({100 * len(bad_voltage) / len(cohort_sample):.3f}%)")

# 6. Cohort metadata

In [ ]:
# ---------------------------------------------------------------------------
# Cohort metadata 
# ---------------------------------------------------------------------------
print(cohort_meta[['state', 'dnsp_name', 'manufacturer']].describe(include='object'))

print("\n=== Capacity distributions (cohort) ===")
print(cohort_meta[['dc_capacity_kw', 'ac_capacity_kw', 'export_limit_kw', 's_99']].describe())

print("\n=== Monitoring window ===")
print(cohort_meta[['monitoring_start', 'first_seen', 'last_seen']].describe())

# 7. Cohort restrict - Random sites

In [ ]:
# ---------------------------------------------------------------------------
# Site-level sample ~100 random cohort sites, ALL their rows.
# ---------------------------------------------------------------------------
N_SAMPLE_SITES = 100
RANDOM_STATE = 42
SAMPLE_YEAR = 2024
SAMPLE_MONTH = 9

sample_site_ids = cohort_meta['site_id'].sample(
    n=min(N_SAMPLE_SITES, len(cohort_meta)), random_state=RANDOM_STATE
)
con.register("sample_site_ids_df", sample_site_ids.to_frame())

site_sample = con.sql(f"""
    SELECT m.*
    FROM ami_meter_residential m
    JOIN sample_site_ids_df s USING (site_id)
    WHERE m.year = {SAMPLE_YEAR} AND m.month = {SAMPLE_MONTH}
    ORDER BY m.site_id, m.t_stamp
""").df()

print(f"Site sample ({SAMPLE_YEAR}-{SAMPLE_MONTH:02d}): {len(sample_site_ids):,} sites, "
      f"{len(site_sample):,} rows ({len(site_sample) / len(sample_site_ids):,.0f} rows/site on average)")

print("\nRows per site (this sample):")
print(site_sample.groupby('site_id').size().describe())

In [ ]:
##################
### OBS: some cohort sites showed partial coverage in the rows_per_site distribution.
### THese sites may not have any data in the month we sampled, so check for that.
##################

sites_missing_month = set(sample_site_ids) - set(site_sample['site_id'].unique())
print(f"Sampled sites with NO data in {SAMPLE_YEAR}-{SAMPLE_MONTH:02d}: {len(sites_missing_month)}")

In [ ]:
site_sample['V'].describe()

# 8. Query ground truth

In [ ]:
# ---------------------------------------------------------------------------
# Register the ami_raw_phaseseparate view (ground truth, circuit-level, load AND PV rows).
# ---------------------------------------------------------------------------
con.sql(f"""
    CREATE OR REPLACE VIEW ami_raw_phaseseparate AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_raw_phaseseparate").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")

In [ ]:
# ---------------------------------------------------------------------------
# Check sample sites are actually present in ami_raw_phaseseparate before pulling
# ami_meter_residential and ami_raw_phaseseparate don't share an identical site_id set
# ---------------------------------------------------------------------------
sample_meta = cohort_meta.loc[cohort_meta['site_id'].isin(sample_site_ids)]
print(sample_meta['in_ami_raw_phaseseparate'].value_counts())

missing_from_raw = sample_meta.loc[~sample_meta['in_ami_raw_phaseseparate'], 'site_id']
if len(missing_from_raw):
    print(f"\n{len(missing_from_raw)} of your {len(sample_site_ids)} sample sites are flagged "
          f"NOT in ami_raw_phaseseparate -- they'll be absent from the ground-truth pull below: "
          f"{missing_from_raw.tolist()}")

In [ ]:
# ---------------------------------------------------------------------------
# Pull the ground-truth data for the SAME sites and SAME month as site_sample, so the two DataFrames are directly comparable.
# ---------------------------------------------------------------------------
raw_phaseseparate_sample = con.sql(f"""
    SELECT r.*
    FROM ami_raw_phaseseparate r
    JOIN sample_site_ids_df s USING (site_id)
    WHERE r.year = {SAMPLE_YEAR} AND r.month = {SAMPLE_MONTH}
    ORDER BY r.site_id, r.circuit_type, r.t_stamp
""").df()

print(f"ami_raw_phaseseparate sample ({SAMPLE_YEAR}-{SAMPLE_MONTH:02d}): "
      f"{len(raw_phaseseparate_sample):,} rows, "
      f"{raw_phaseseparate_sample['site_id'].nunique():,} / {len(sample_site_ids)} sample sites present")

print("\nRows by circuit_type:")
print(raw_phaseseparate_sample['circuit_type'].value_counts())

print("\nRows per site (ground truth):")
print(raw_phaseseparate_sample.groupby('site_id').size().describe())

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Compare synthetic meter (site_sample) against ground truth
# (raw_phaseseparate_sample), for the ac_load_net circuits only -- these are
# the SAME physical circuit reading in both tables. ami_meter_residential's
# P_kw/Q_kvar are RAW (uncorrected sign); ami_raw_phaseseparate's
# P_kw_signed/Q_kvar_signed are polarity-corrected via circuit_polarity.
# V has no sign-convention ambiguity, so it should match exactly either way.
# ---------------------------------------------------------------------------

ground_truth_load = raw_phaseseparate_sample.loc[
    raw_phaseseparate_sample['circuit_type'] == 'ac_load_net'
].copy()

compare = site_sample.merge(
    ground_truth_load,
    on=['site_id', 'device_id', 'circuit_id', 't_stamp'],
    how='inner',
    suffixes=('_meter', '_raw'),
)

print(f"site_sample rows:          {len(site_sample):,}")
print(f"ground_truth_load rows:    {len(ground_truth_load):,}")
print(f"Matched on keys:           {len(compare):,}")

unmatched = len(site_sample) - len(compare)
if unmatched:
    print(f"\n{unmatched:,} site_sample rows had NO matching ground-truth row on "
          f"(site_id, device_id, circuit_id, t_stamp) -- worth investigating before "
          f"trusting the rest of this comparison.")

# --- Voltage: no polarity ambiguity, should match exactly ---
compare['V_diff'] = compare['V_meter'] - compare['V_raw']
print("\n=== Voltage diff (V_meter - V_raw) ===")
print(compare['V_diff'].describe())

# --- Active power: check magnitude match and sign relationship ---
compare['P_kw_abs_diff'] = (compare['P_kw'].abs() - compare['P_kw_signed'].abs()).abs()
compare['P_kw_same_sign'] = (compare['P_kw'] * compare['P_kw_signed']) >= 0

print("\n=== |P_kw| vs |P_kw_signed| absolute difference ===")
print(compare['P_kw_abs_diff'].describe())

print("\n=== Sign agreement (P_kw vs P_kw_signed) ===")
print(compare['P_kw_same_sign'].value_counts())
print(f"(circuit_polarity flips sign when this is False, consistently or not "
      f"-- check whether disagreement clusters by circuit_id below)")

# --- Reactive power: same pattern ---
compare['Q_kvar_abs_diff'] = (compare['Q_kvar'].abs() - compare['Q_kvar_signed'].abs()).abs()
print("\n=== |Q_kvar| vs |Q_kvar_signed| absolute difference ===")
print(compare['Q_kvar_abs_diff'].describe())

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Does sign (dis)agreement cluster by circuit -- i.e. is
# circuit_polarity a fixed per-circuit_id property (expected), or does it
# flip within a circuit over time (would indicate a real problem)?
# ---------------------------------------------------------------------------
sign_by_circuit = compare.groupby('circuit_id')['P_kw_same_sign'].agg(['mean', 'nunique'])
sign_by_circuit.columns = ['pct_same_sign', 'n_distinct_values']

print("Circuits where sign agreement is NOT constant over time (potential issue):")
print(sign_by_circuit.loc[sign_by_circuit['n_distinct_values'] > 1])

print(f"\n{len(sign_by_circuit)} circuits total; "
      f"{(sign_by_circuit['n_distinct_values'] == 1).sum()} have a fixed sign relationship "
      f"throughout the sample (expected if circuit_polarity is a static per-circuit flag).")

# 9. Correct polarity

In [ ]:
# ---------------------------------------------------------------------------
# Cell: imports + polarity-corrected ami_meter_residential view, matching
# what 05_conformance_net_meter_blind actually plotted from (NOT the raw
# pass-through view we registered earlier in this notebook -- that one is
# deliberately uncorrected, fine for EDA on raw signal, but wrong for
# reusing the Volt-VAr classification pipeline, which assumes correct sign).
# ---------------------------------------------------------------------------


# Keep the existing raw view under its own name; build the corrected one on
# top of it into a NEW view name so we don't lose the raw version we've
# already been using for the rest of this notebook's EDA.
con.sql(f"""
    CREATE OR REPLACE VIEW ami_circuit_metadata AS
    SELECT * FROM read_parquet('{Config.store_path("ami_circuit_metadata").as_posix()}')
""")

con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_residential_polarity_corrected AS
    {Polarity.build_polarity_corrected_sql("ami_meter_residential", "ami_circuit_metadata")}
""")

print("ami_meter_residential_polarity_corrected ready.")

# 10. Check Volt-Var scatter

In [ ]:
# Example of major surplus:
# EXAMPLE_SITE_ID = int(sample_site_ids.iloc[3])

SITE_ID = 1084068901
EXAMPLE_SITE_ID = SITE_ID

In [ ]:
# ---------------------------------------------------------------------------
# The four Volt-VAr data variants
#
#   Polarity-corrected only
#     source_table = ami_meter_residential_polarity_corrected
#     circuit_col  = load_circuit_id   (net household circuit, PV+load combined)
#     q_col        = Q_kvar            (raw net reactive power)
#     use_baseline_correction = False
#     notebook vars: pv_scope_classified_raw / rating_kw_raw
#
#   Sign-flip only
#     source_table = ami_meter_residential_sign_flipped
#     circuit_col  = load_circuit_id   (net household circuit)
#     q_col        = Q_kvar            (raw net reactive power)
#     use_baseline_correction = False
#     notebook vars: pv_scope_classified_flipped / rating_kw_flipped
#
#   Polarity-corrected + baseline estimate
#     source_table = ami_meter_residential_polarity_corrected  (same source as above)
#     circuit_col  = load_circuit_id   (net household circuit)
#     q_col        = Q_kvar_pv_est     (baseline household-load estimate subtracted,
#                                        isolating the PV-attributable Q)
#     use_baseline_correction = True
#     notebook vars: pv_scope_classified / rating_kw
#
#   Ground truth (ami_raw_phaseseparate)
#     source_table = ami_raw_phaseseparate_pv_only
#     circuit_col  = pv_circuit_id     (physically isolated PV-only circuit, not net)
#     q_col        = Q_kvar            (raw -- no baseline correction needed,
#                                        already isolated from load)
#     use_baseline_correction = False
#     notebook vars: pv_scope_classified_true / rating_kw_true
#
# The first three vary along two independent axes: which sign-handling fix
# was applied upstream to the net signal (polarity-corrected vs sign-flipped
# -- two different resolutions of the same load/generator-convention
# ambiguity), and whether a baseline household-load estimate has been
# subtracted to isolate PV-attributable Q (use_baseline_correction). Ground
# truth sits outside that grid entirely: pv_circuit_id is a physically
# separated PV circuit, so there's no net-signal sign or load-contamination
# question to resolve, which is why it serves as the reference for the
# other three.
# ---------------------------------------------------------------------------

## 10.1. Direct AMI reading

In [ ]:
example_circuit_id = int(
    site_sample.loc[site_sample['site_id'] == EXAMPLE_SITE_ID, 'circuit_id'].iloc[0]
)

scope_where = f"(year = {SAMPLE_YEAR} AND month = {SAMPLE_MONTH})"
scope_label = f"{SAMPLE_YEAR}-{SAMPLE_MONTH:02d}"


fig_scatter_raw, pv_scope_classified_raw, rating_kw_raw = ConformancePlot.plot_pv_circuit_vvar_scatter(
    con, EXAMPLE_SITE_ID, example_circuit_id,
    scope_where=scope_where,
    period_label=f"{scope_label} (polarity-corrected net meter, no baseline correction)",
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    source_table="ami_meter_residential_polarity_corrected",
    metadata_table="ami_site_metadata_residential",
    use_baseline_correction=False,  # raw net reading, not the PV-estimated signal
    figsize=(8, 6),
)

## 10.2. Signal corrected

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW ami_meter_residential_sign_flipped AS
    SELECT
        * EXCLUDE (P_kw, Q_kvar),
        -P_kw AS P_kw,
        -Q_kvar AS Q_kvar
    FROM ami_meter_residential_polarity_corrected
""")

fig_scatter_flipped, pv_scope_classified_flipped, rating_kw_flipped = ConformancePlot.plot_pv_circuit_vvar_scatter(
    con, EXAMPLE_SITE_ID, example_circuit_id,
    scope_where=scope_where,
    period_label=f"{scope_label} (sign-flip only, no baseline offset)",
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    source_table="ami_meter_residential_sign_flipped",
    metadata_table="ami_site_metadata_residential",
    use_baseline_correction=False,  # the view already did the sign flip; no baseline on top
    figsize=(8, 6),
)

## 10.3. Baseline correction

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Reproduce the Q-vs-V scatter, scoped to our SAMPLE_YEAR/SAMPLE_MONTH,
# for one site from our cohort sample -- reusing plot_pv_circuit_vvar_scatter
# unmodified. Since our cohort is 1-load-phase by construction, each sample
# site has exactly one ac_load_net circuit_id -- grab it straight from
# site_sample rather than guessing.
# ---------------------------------------------------------------------------
example_circuit_id = int(
    site_sample.loc[site_sample['site_id'] == EXAMPLE_SITE_ID, 'circuit_id'].iloc[0]
)

scope_where = f"(year = {SAMPLE_YEAR} AND month = {SAMPLE_MONTH})"
scope_label = f"{SAMPLE_YEAR}-{SAMPLE_MONTH:02d}"

fig_scatter, pv_scope_classified, rating_kw = ConformancePlot.plot_pv_circuit_vvar_scatter(
    con, EXAMPLE_SITE_ID, example_circuit_id,
    scope_where=scope_where,
    period_label=scope_label,
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    source_table="ami_meter_residential_polarity_corrected",
    metadata_table="ami_site_metadata_residential",
    # use_baseline_correction=True by default -- same "blind" (net-meter,
    # nighttime-baseline-estimated PV signal) approach as notebook 05.
    figsize=(8, 6),
)

In [ ]:
# ---------------------------------------------------------------------------
# Cell: The actual baseline correction applied to this site/circuit/scope.
# pv_scope_classified already carries nighttime_baseline_p_kw/q_kvar as a
# constant column (one value, resolved once over the whole scope_where
# window -- see the "monthly, not per-day" note from earlier). The
# transformation itself is pv_estimate = baseline - net_reading (a sign
# flip PLUS an offset, not a plain subtraction of a constant).
# ---------------------------------------------------------------------------
baseline_p_kw = pv_scope_classified['nighttime_baseline_p_kw'].iloc[0]
baseline_q_kvar = pv_scope_classified['nighttime_baseline_q_kvar'].iloc[0]

print(f"Nighttime baseline used for site_id={EXAMPLE_SITE_ID}, circuit_id={example_circuit_id}, "
      f"scope={scope_label}:")
print(f"  baseline_p_kw   = {baseline_p_kw:+.4f} kW   ({100*baseline_p_kw/rating_kw:+.2f}% of {rating_kw:.2f} kW rating)")
print(f"  baseline_q_kvar = {baseline_q_kvar:+.4f} kvar ({100*baseline_q_kvar/rating_kw:+.2f}% of rating)")

print(f"\nTransformation applied to every interval: "
      f"P_kw_pv_est = {baseline_p_kw:+.4f} - P_kw   (i.e. raw net reading negated, then offset)")

# Compare raw net reading vs the resulting estimate, so the actual effect
# size is visible, not just the constant.
comparison = pv_scope_classified[['t_stamp', 'P_kw', 'P_kw_pv_est', 'Q_kvar', 'Q_kvar_pv_est']].copy()
comparison['P_kw_shift'] = comparison['P_kw_pv_est'] - (-comparison['P_kw'])  # should equal baseline_p_kw exactly
comparison['Q_kvar_shift'] = comparison['Q_kvar_pv_est'] - (-comparison['Q_kvar'])

print("\nSanity check -- the residual shift after accounting for the sign flip should be "
      "EXACTLY the baseline constant for every row (confirms the formula, not an approximation):")
print(comparison[['P_kw_shift', 'Q_kvar_shift']].describe())

## 10.4. Ground truth

In [ ]:
# ---------------------------------------------------------------------------
# Cell: TRUE version of the same scatter, from ami_raw_phaseseparate's
# already-separated pv_site_net signal -- no baseline estimation needed,
# since this is the real PV reading, not an inference from the net meter.
# Alias columns to match what plot_pv_circuit_vvar_scatter/build_voltvar_sql
# expect (V, P_kw, Q_kvar), since ami_raw_phaseseparate names them
# differently (P_kw_signed/Q_kvar_signed) and carries both circuit_types.
# ---------------------------------------------------------------------------
con.sql("""
    CREATE OR REPLACE VIEW ami_raw_phaseseparate_pv_only AS
    SELECT
        site_id, device_id, circuit_id, t_stamp, year, month,
        V,
        P_kw_signed AS P_kw,
        Q_kvar_signed AS Q_kvar
    FROM ami_raw_phaseseparate
    WHERE circuit_type = 'pv_site_net'
""")

# The site's TRUE pv_site_net circuit_id, from the ground-truth sample we
# already pulled -- NOT the ac_load_net circuit_id used above.
true_pv_circuit_id = int(
    raw_phaseseparate_sample.loc[
        (raw_phaseseparate_sample['site_id'] == EXAMPLE_SITE_ID)
        & (raw_phaseseparate_sample['circuit_type'] == 'pv_site_net'),
        'circuit_id'
    ].iloc[0]
)

fig_scatter_true, pv_scope_classified_true, rating_kw_true = ConformancePlot.plot_pv_circuit_vvar_scatter(
    con, EXAMPLE_SITE_ID, true_pv_circuit_id,
    scope_where=scope_where,
    period_label=f"{scope_label} (true PV, ami_raw_phaseseparate)",
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    source_table="ami_raw_phaseseparate_pv_only",
    metadata_table="ami_site_metadata_residential",
    use_baseline_correction=False,  # already the real, separated PV signal
    figsize=(8, 6),
)

## 10.5. All compared

In [ ]:
# ---------------------------------------------------------------------------
# Cell: 4-panel side-by-side -- polarity-corrected only / sign-flip only /
# baseline-estimated / ground truth. Same rasterize-and-place approach,
# since plot_pv_circuit_vvar_scatter always builds its own standalone figure.
# ---------------------------------------------------------------------------


fig_combined, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(28, 8), dpi=130)

panels = [
    (ax1, fig_scatter_raw, f"Polarity-corrected only — {scope_label}"),
    (ax2, fig_scatter_flipped, f"Sign-flip only, no baseline — {scope_label}"),
    (ax3, fig_scatter, f"Polarity-corrected + baseline estimate — {scope_label}"),
    (ax4, fig_scatter_true, f"Ground truth (ami_raw_phaseseparate) — {scope_label}"),
]

for ax, source_fig, title in panels:
    buf = io.BytesIO()
    source_fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    image = mpimg.imread(buf)
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(title, fontsize=11)

# fig_combined.suptitle(f"Site {EXAMPLE_SITE_ID}: Volt-VAr response across four data variants", fontsize=13)
fig_combined.tight_layout()

## 10.6. Monthly view for 1 site, day by day

In [ ]:
fig3d_flipped, ax3d_flipped, window_flipped_3d = ConformancePlot.plot_assessable_vvar_scatter3d(
    pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar",
    title_suffix=f" — sign-flip only, {scope_label}",
    daytime_hours=None,

    ols_grouping=None,   # or "scope" / "day" / "week" / "weekday_weekend" / None / "surface"
    box_aspect=(1.0, 3.0, 1.0), # (Voltage, Time, Reactive power) — time stretched 4x

    show_required_plane=False,
    show_colorbar=False,   # <-- omit the colorbar
    elev=15, azim=-60,
)

In [ ]:
fig3d_flipped_interactive = ConformancePlot.plot_assessable_vvar_scatter3d_interactive(
    pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar",
    title_suffix=f" — sign-flip only, {scope_label}",
    daytime_hours=None,
    ols_grouping="day",
    aspect_ratio=(1.0, 3.0, 1.0),
    show_required_plane=False,
    show_colorbar=False,
)
fig3d_flipped_interactive.show()

# 11. Check daily responses

In [ ]:
# ---------------------------------------------------------------------------
# Cell: 3x4 grid -- one day, same site/circuit, comparing the 4 variants
# (columns) across V / P / Q (rows). Reuses the already-computed
# pv_scope_classified* DataFrames, filtered down to one AEST calendar day.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

EXAMPLE_DATE = f"{SAMPLE_YEAR}-{SAMPLE_MONTH:02d}-22"  # pick any day in the sample month

# AEST calendar day -> [start, end) in the raw t_stamp storage range,
# same convention used throughout this codebase (t_stamp + 10h = AEST, no DST).
day_start_utc = pd.Timestamp(EXAMPLE_DATE) - pd.Timedelta(hours=10)
day_end_utc = day_start_utc + pd.Timedelta(days=1)

def _day_slice(df):
    return df.loc[(df['t_stamp'] >= day_start_utc) & (df['t_stamp'] < day_end_utc)].sort_values('t_stamp')

# Each variant's classified DataFrame carries P/Q under different column
# names depending on whether baseline correction was applied -- V is
# unaffected either way. The sign-flipped variant already has P_kw/Q_kvar
# pre-negated by the view itself, so it uses the same column names as raw.
variants = [
#    ("Polarity-corrected only\n(no baseline)", _day_slice(pv_scope_classified_raw), "P_kw", "Q_kvar"),
    ("Sign-flip only\n(no baseline)", _day_slice(pv_scope_classified_flipped), "P_kw", "Q_kvar"),
    ("Polarity-corrected +\nbaseline estimate", _day_slice(pv_scope_classified), "P_kw_pv_est", "Q_kvar_pv_est"),
    ("Ground truth\n(ami_raw_phaseseparate)", _day_slice(pv_scope_classified_true), "P_kw", "Q_kvar"),
]

fig, axes = plt.subplots(3, 3, figsize=(24, 11), sharex=True, sharey='row', dpi=120)

for col_idx, (title, d, p_col, q_col) in enumerate(variants):
    axes[0, col_idx].set_title(title, fontsize=10.5, fontweight="bold")

    axes[0, col_idx].plot(d['t_stamp'], d['V'], color="#2a78d6", linewidth=1)
    axes[1, col_idx].plot(d['t_stamp'], d[p_col], color="#eb6834", linewidth=1)
    axes[2, col_idx].plot(d['t_stamp'], d[q_col], color="#1baf7a", linewidth=1)
    axes[2, col_idx].axhline(0, color="black", linewidth=0.5)

    axes[2, col_idx].set_xlabel("Time (UTC)", fontsize=9)
    axes[2, col_idx].tick_params(axis='x', rotation=45)

for row_idx, ylabel in enumerate(["Voltage (V)", "Active power (kW)", "Reactive power (kvar)"]):
    axes[row_idx, 0].set_ylabel(ylabel, fontsize=9)

fig.suptitle(
    f"Site {EXAMPLE_SITE_ID}, circuit {example_circuit_id} | {EXAMPLE_DATE} (AEST day)",
    fontsize=13, fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 1])

In [ ]:
# Site 376404014, circuit 614271 (load) / 614272 (true PV), day 2024-09-17 (AEST)
# Reuses EXAMPLE_SITE_ID / example_circuit_id / true_pv_circuit_id / day_start_utc / day_end_utc
# from your 3x4 day-grid cell — if those aren't still set to this site, set them explicitly first:
# EXAMPLE_SITE_ID = 376404014
# example_circuit_id = 614271
# true_pv_circuit_id = 614272

flipped_day = con.execute("""
    SELECT t_stamp, V, P_kw, Q_kvar
    FROM ami_meter_residential_sign_flipped
    WHERE circuit_id = ?
      AND t_stamp >= ?
      AND t_stamp <  ?
    ORDER BY t_stamp
""", [example_circuit_id, day_start_utc, day_end_utc]).df()

gt_pv_day = con.execute("""
    SELECT t_stamp, V, P_kw, Q_kvar
    FROM ami_raw_phaseseparate_pv_only
    WHERE circuit_id = ?
      AND t_stamp >= ?
      AND t_stamp <  ?
    ORDER BY t_stamp
""", [true_pv_circuit_id, day_start_utc, day_end_utc]).df()

print(f"flipped_day: {len(flipped_day)} rows, gt_pv_day: {len(gt_pv_day)} rows")

corr_netQ_netP = flipped_day[["P_kw", "Q_kvar"]].corr().iloc[0, 1]
corr_netP_V   = flipped_day[["P_kw", "V"]].corr().iloc[0, 1]
corr_gt_QP    = gt_pv_day[["P_kw", "Q_kvar"]].corr().iloc[0, 1]

print(f"net circuit  Q~P corr: {corr_netQ_netP:.3f}")
print(f"net circuit  P~V corr: {corr_netP_V:.3f}")
print(f"true PV circ Q~P corr: {corr_gt_QP:.3f}")

# 12. Slope checking

## 12.1. sign-flip only

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Assessable Volt-VAr window, restricted to 10am-4pm AEST (daytime,
# when PV is likeliest to be generating enough to actually be assessable).
# ---------------------------------------------------------------------------

# Below would plot the " Sign-flip only"
'''
slope_emp_flipped_day, intercept_emp_flipped_day, window_flipped_day, fit_stats_flipped_day = ConformancePlot.plot_assessable_vvar_window(
    pv_scope_classified_true, rating_kw_flipped, "Q_kvar",
    title_suffix=f" — sign-flip only, {scope_label}",
    daytime_hours=None,
)
'''
# ---------------------------------------------------------------------------
# Cell: Assessable Volt-VAr window, restricted to 10am-4pm AEST (daytime,
# when PV is likeliest to be generating enough to actually be assessable).
# ---------------------------------------------------------------------------
slope_emp_baseline_day, intercept_emp_baseline_day, window_baseline_day, fit_stats_baseline_day = ConformancePlot.plot_assessable_vvar_window(
    pv_scope_classified, rating_kw, "Q_kvar_pv_est",
    title_suffix=f"\nPolarity-corrected + baseline estimate, {scope_label}",
    daytime_hours=None,
)

## 10.2. All variants

In [ ]:
# ---------------------------------------------------------------------------
# Cell: All 4 variants' assessable Volt-VAr window, side by side as
# subplots -- reusing plot_assessable_vvar_window's own `ax` parameter, so
# each panel is drawn directly into a shared figure (no rasterize/combine
# trick needed here, unlike the Q-vs-V scatter cells earlier, since this
# function was written from the start to accept an external ax).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(28, 7), dpi=130)

variant_specs = [
    (axes[0], pv_scope_classified_raw, rating_kw_raw, "Q_kvar", "Polarity-corrected only"),
    (axes[1], pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar", "Sign-flip only"),
    (axes[2], pv_scope_classified, rating_kw, "Q_kvar_pv_est", "Polarity-corrected + baseline estimate"),
    (axes[3], pv_scope_classified_true, rating_kw_true, "Q_kvar", "Ground truth (ami_raw_phaseseparate)"),
]

slopes = {}
for ax, classified_df, rating, q_col, label in variant_specs:
    slope_emp, intercept_emp, window, fit_stats = ConformancePlot.plot_assessable_vvar_window(
        classified_df, rating, q_col,
        title_suffix=f" — {label}",
        ax=ax,
    )
    slopes[label] = slope_emp

fig.suptitle(
    f"Site {EXAMPLE_SITE_ID}: assessable Volt-VAr window across four data variants — {scope_label}",
    fontsize=13, fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.95])

print("\nEmpirical slope (%rating/V) by variant, vs required "
      f"{ConformancePlot._VVAR_ABSORB_SLOPE_PCT_PER_V:.3f} %/V:")
for label, slope in slopes.items():
    print(f"  {label:<45} {slope:+.3f} %/V")

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Last 3 variants' assessable Volt-VAr window, side by side as
# subplots (Polarity-corrected only omitted) -- reusing
# plot_assessable_vvar_window's own `ax` parameter, so each panel is drawn
# directly into a shared figure.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(21, 7), dpi=130)

variant_specs = [
    (axes[0], pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar", "Sign-flip only"),
    (axes[1], pv_scope_classified, rating_kw, "Q_kvar_pv_est", "Polarity-corrected + baseline estimate"),
    (axes[2], pv_scope_classified_true, rating_kw_true, "Q_kvar", "Ground truth (ami_raw_phaseseparate)"),
]

slopes = {}
for ax, classified_df, rating, q_col, label in variant_specs:
    slope_emp, intercept_emp, window, fit_stats = ConformancePlot.plot_assessable_vvar_window(
        classified_df, rating, q_col,
        title_suffix=f" | {label}",
        ax=ax,
    )
    slopes[label] = slope_emp

fig.suptitle(
    f"Site {EXAMPLE_SITE_ID}: assessable Volt-VAr window across three data variants | {scope_label}",
    fontsize=13, fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.99])

print("\nEmpirical slope (%rating/V) by variant, vs required "
      f"{ConformancePlot._VVAR_ABSORB_SLOPE_PCT_PER_V:.3f} %/V:")
for label, slope in slopes.items():
    print(f"  {label:<45} {slope:+.3f} %/V")

## 10.3. Time constrained

In [ ]:
# ---------------------------------------------------------------------------
# Cell: All 4 variants' assessable Volt-VAr window, side by side as
# subplots -- reusing plot_assessable_vvar_window's own `ax` parameter, so
# each panel is drawn directly into a shared figure (no rasterize/combine
# trick needed here, unlike the Q-vs-V scatter cells earlier, since this
# function was written from the start to accept an external ax).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(28, 7), dpi=130)

DAY_TIME_HOURS = (10, 16)  # AEST 10am-4pm, same as the previous cell

variant_specs = [
    (axes[0], pv_scope_classified_raw, rating_kw_raw, "Q_kvar", "Polarity-corrected only"),
    (axes[1], pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar", "Sign-flip only"),
    (axes[2], pv_scope_classified, rating_kw, "Q_kvar_pv_est", "Polarity-corrected + baseline estimate"),
    (axes[3], pv_scope_classified_true, rating_kw_true, "Q_kvar", "Ground truth (ami_raw_phaseseparate)"),
]

slopes = {}
for ax, classified_df, rating, q_col, label in variant_specs:
    slope_emp, intercept_emp, window, fit_stats = ConformancePlot.plot_assessable_vvar_window(
        classified_df, rating, q_col,
        title_suffix=f" — {label}",
        ax=ax,
        daytime_hours=DAY_TIME_HOURS
    )
    slopes[label] = slope_emp

fig.suptitle(
    f"Site {EXAMPLE_SITE_ID}: assessable Volt-VAr window across four data variants — {scope_label}",
    fontsize=13, fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.95])

print("\nEmpirical slope (%rating/V) by variant, vs required "
      f"{ConformancePlot._VVAR_ABSORB_SLOPE_PCT_PER_V:.3f} %/V:")
for label, slope in slopes.items():
    print(f"  {label:<45} {slope:+.3f} %/V")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(28, 7), dpi=130)

DAY_TIME_HOURS = (10, 16)  # AEST 10am-4pm, same as the previous cell
V_MIN, V_MAX = 242, 253    # trim the noisy bottom of the ramp near V3=240

variant_specs = [
    (axes[0], pv_scope_classified_raw, rating_kw_raw, "Q_kvar", "Polarity-corrected only"),
    (axes[1], pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar", "Sign-flip only"),
    (axes[2], pv_scope_classified, rating_kw, "Q_kvar_pv_est", "Polarity-corrected + baseline estimate"),
    (axes[3], pv_scope_classified_true, rating_kw_true, "Q_kvar", "Ground truth (ami_raw_phaseseparate)"),
]

slopes = {}
for ax, classified_df, rating, q_col, label in variant_specs:
    slope_emp, intercept_emp, window, fit_stats = ConformancePlot.plot_assessable_vvar_window(
        classified_df, rating, q_col,
        title_suffix=f" — {label}",
        ax=ax,
        daytime_hours=DAY_TIME_HOURS,
        v_min=V_MIN, v_max=V_MAX,
    )
    slopes[label] = slope_emp

fig.suptitle(
    f"Site {EXAMPLE_SITE_ID}: assessable Volt-VAr window across four data variants — {scope_label}\n"
    f"(fitted over [{V_MIN}, {V_MAX}) V)",
    fontsize=13, fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.95])

print("\nEmpirical slope (%rating/V) by variant, vs required "
      f"{ConformancePlot._VVAR_ABSORB_SLOPE_PCT_PER_V:.3f} %/V:")
for label, slope in slopes.items():
    print(f"  {label:<45} {slope:+.3f} %/V")

## 12. Sign gated

In [ ]:
slope_emp_baseline_day, intercept_emp_baseline_day, window_baseline_day, fit_stats_baseline_day = ConformancePlot.plot_assessable_vvar_window(
    pv_scope_classified, 
    rating_kw, 
    "Q_kvar_pv_est",
    title_suffix=f"\nPolarity-corrected + baseline estimate (signed gate), {scope_label}",
    daytime_hours=None,
    power_col="P_kw_pv_est",
)

# 11. Cohort view

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Build the site -> circuit_id map for the whole sample cohort.
# load_circuit_id covers the first 3 variants (all views of the net-meter
# family); pv_circuit_id (from ground truth) covers the 4th. A site missing
# from raw_phaseseparate_sample gets pv_circuit_id = NaN -- handled, not
# silently dropped, by fetch_cohort_assessable_vvar_slopes.
# ---------------------------------------------------------------------------
load_circuits = (
    site_sample.drop_duplicates('site_id')[['site_id', 'circuit_id']]
    .rename(columns={'circuit_id': 'load_circuit_id'})
)
pv_circuits = (
    raw_phaseseparate_sample.loc[raw_phaseseparate_sample['circuit_type'] == 'pv_site_net']
    .drop_duplicates('site_id')[['site_id', 'circuit_id']]
    .rename(columns={'circuit_id': 'pv_circuit_id'})
)
site_circuit_map = load_circuits.merge(pv_circuits, on='site_id', how='left')

print(f"{len(site_circuit_map):,} sites in map; "
      f"{site_circuit_map['pv_circuit_id'].isna().sum():,} missing a ground-truth PV circuit.")

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Fetch the fit for every (site, variant) pair across the sample
# cohort. This re-queries per site/variant (no shortcut around that -- each
# needs its own baseline/classification pass), so expect this to take a
# while for ~100 sites x 4 variants.
# ---------------------------------------------------------------------------
cohort_vvar_fits = ConformancePlot.fetch_cohort_assessable_vvar_slopes(
    con, site_circuit_map,
    scope_where=scope_where,
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    metadata_table="ami_site_metadata_residential",
)
cohort_vvar_fits['status'].value_counts()

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Cohort-wide slope comparison plot.
# ---------------------------------------------------------------------------
plotted_fits = ConformancePlot.plot_cohort_assessable_vvar_slopes(cohort_vvar_fits, min_n_window=20)

In [ ]:
# ---------------------------------------------------------------------------
# Cell: All ~65 sites' individual OLS lines, faceted by variant.
# ---------------------------------------------------------------------------

### ALL VARIANTS:
#fig_ols_lines = ConformancePlot.plot_cohort_ols_lines(plotted_fits)

# ---------------------------------------------------------------------------
# Polarity-corrected + baseline estimate and Ground truth only.
# ---------------------------------------------------------------------------
VARIANTS_SUBSET = ["Polarity-corrected + baseline estimate", "Ground truth (ami_raw_phaseseparate)"]
PALETTE_SUBSET = ["#1baf7a", "#eda100"]

fig_ols_lines = ConformancePlot.plot_cohort_ols_lines(
    plotted_fits, variant_order=VARIANTS_SUBSET, palette=PALETTE_SUBSET,
    figsize=(12, 6)
)

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Pass/fail split by slope tolerance -- swap TOLERANCE_PCT to try
# 10 / 20 / 30 (etc.) and see how the pass rate shifts per variant.
# (unchanged: this still drives the 2x4 PASSED/FAILED plot below, using the
# original symmetric slope test.)
# ---------------------------------------------------------------------------
TOLERANCE_PCT = 20

VARIANTS_SUBSET = ["Polarity-corrected + baseline estimate", "Ground truth (ami_raw_phaseseparate)"]
PALETTE_SUBSET = ["#1baf7a", "#eda100"]

flagged_fits = ConformancePlot.plot_cohort_ols_lines_pass_fail(
    plotted_fits, TOLERANCE_PCT, variant_order=VARIANTS_SUBSET, palette=PALETTE_SUBSET,
    figsize=(12, 6)
)

print(f"\nPass rate by variant (±{TOLERANCE_PCT}% of required slope):")
print((flagged_fits.groupby('variant')['passed'].mean() * 100).round(1).rename('% passed'))

# ... rest of the cell unchanged ...
# ---------------------------------------------------------------------------
# level/offset test -- does the OLS line sit close to the required
# line, not just have the right slope? Computed at V3=240, V4=258, and
# their midpoint=249 (over the full Volt-VAr ramp, not the narrower
# assessable window). Only the midpoint gates pass/fail for now; all three
# are kept as columns for later querying.
#
# Also demonstrates the new asymmetric slope band (independent of the
# symmetric TOLERANCE_PCT test above -- both live side by side in
# flagged_fits).
# ---------------------------------------------------------------------------
NEG_TOLERANCE_PCT = -20   # slope allowed up to 10% *weaker* (less negative) than required
POS_TOLERANCE_PCT = 1000   # slope allowed up to 100% *stronger* (more negative) than required

OFFSET_NEG_TOLERANCE_PP = -4.0   # midpoint offset allowed 4pp below the required line
OFFSET_POS_TOLERANCE_PP = 4.0    # midpoint offset allowed 4pp above the required line

flagged_fits = ConformancePlot.add_offset_pct_columns(flagged_fits)
flagged_fits = ConformancePlot.add_slope_pass_fail_asymmetric(
    flagged_fits, neg_tolerance_pct=NEG_TOLERANCE_PCT, pos_tolerance_pct=POS_TOLERANCE_PCT,
)
flagged_fits = ConformancePlot.add_offset_pass_fail(
    flagged_fits, neg_tolerance_pp=OFFSET_NEG_TOLERANCE_PP, pos_tolerance_pp=OFFSET_POS_TOLERANCE_PP,
)

print(f"\nPass rate by variant (asymmetric slope band {NEG_TOLERANCE_PCT:+.0f}%/{POS_TOLERANCE_PCT:+.0f}% "
      f"of required slope):")
print((flagged_fits.groupby('variant')['slope_passed_asym'].mean() * 100).round(1).rename('% passed (slope, asym)'))

print(f"\nPass rate by variant (offset at midpoint=249V, {OFFSET_NEG_TOLERANCE_PP:+.1f}pp/"
      f"{OFFSET_POS_TOLERANCE_PP:+.1f}pp band):")
print((flagged_fits.groupby('variant')['offset_passed'].mean() * 100).round(1).rename('% passed (offset)'))

# Sites that pass the (symmetric) slope test but fail the offset/level test:
slope_ok_offset_bad = flagged_fits[flagged_fits['passed'] & ~flagged_fits['offset_passed']]
print(f"\n{len(slope_ok_offset_bad)} site x variant rows pass slope but fail offset test")
slope_ok_offset_bad[['site_id', 'variant', 'slope', 'offset_pct_v3', 'offset_pct_mid', 'offset_pct_v4']]

# ---------------------------------------------------------------------------
# Offset test, split further by which slope test it's combined with.
# ---------------------------------------------------------------------------
flagged_fits["passed_offset_and_slope_sym"] = flagged_fits["offset_passed"] & flagged_fits["passed"]
flagged_fits["passed_offset_and_slope_asym"] = flagged_fits["offset_passed"] & flagged_fits["slope_passed_asym"]

combined_rates = pd.concat(
    {
        f"offset AND slope (symmetric ±{TOLERANCE_PCT:.0f}%)":
            flagged_fits.groupby('variant')['passed_offset_and_slope_sym'].mean() * 100,
        f"offset AND slope (asymmetric {NEG_TOLERANCE_PCT:+.0f}%/{POS_TOLERANCE_PCT:+.0f}%)":
            flagged_fits.groupby('variant')['passed_offset_and_slope_asym'].mean() * 100,
    },
    axis=1,
).round(1)

print(f"\nPass rate by variant, offset (midpoint=249V, {OFFSET_NEG_TOLERANCE_PP:+.1f}pp/"
      f"{OFFSET_POS_TOLERANCE_PP:+.1f}pp) combined with each slope test:")
combined_rates

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Verify the ground-truth variant is really using a DIFFERENT circuit
# than the load-based variants -- for the single example site, and for
# every site in the cohort map.
# ---------------------------------------------------------------------------
print(f"Example site {EXAMPLE_SITE_ID}:")
print(f"  load circuit_id (ac_load_net): {example_circuit_id}")
print(f"  true PV circuit_id (pv_site_net): {true_pv_circuit_id}")
print(f"  same circuit? {example_circuit_id == true_pv_circuit_id}")

# Confirm ami_raw_phaseseparate itself agrees these are two distinct circuits
# with the expected circuit_type each.
check = con.sql(f"""
    SELECT DISTINCT circuit_id, circuit_type
    FROM ami_raw_phaseseparate
    WHERE site_id = {EXAMPLE_SITE_ID} AND circuit_id IN ({example_circuit_id}, {true_pv_circuit_id})
""").df()
print("\nCircuit identity check for this site:")
print(check)

# Cohort-wide: no site should have load_circuit_id == pv_circuit_id.
same_circuit = site_circuit_map.loc[site_circuit_map['load_circuit_id'] == site_circuit_map['pv_circuit_id']]
print(f"\nSites where load_circuit_id == pv_circuit_id (should be 0): {len(same_circuit)}")
if len(same_circuit):
    print(same_circuit)

    

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Is ground truth's lower pass rate driven by a genuinely poor linear
# fit (low R^2 -- consistent with fixed-power-factor or non-linear/
# saturating real inverter behavior), or by a consistently-wrong-but-clean
# slope (high R^2, just far from -3.333)? These point to different stories.
# ---------------------------------------------------------------------------
print("Median R^2 and |pct_error| by variant:")
print(flagged_fits.groupby('variant').agg(
    median_r_squared=('r_squared', 'median'),
    median_abs_pct_error=('pct_error', lambda s: s.abs().median()),
    n=('slope', 'count'),
))

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Are the SAME sites passing under ground truth and under the
# derived variants, or are entirely different sites passing? If ground
# truth's passers and (say) baseline-estimate's passers barely overlap,
# that's fairly strong evidence the derived signal's "pass" isn't tracking
# the real inverter's conformance at all -- it's a different, decoupled
# signal that happens to sometimes look compliant.
# ---------------------------------------------------------------------------
pass_by_site = flagged_fits.pivot_table(index='site_id', columns='variant', values='passed', aggfunc='first')

gt_col = 'Ground truth (ami_raw_phaseseparate)'
for other_col in pass_by_site.columns:
    if other_col == gt_col:
        continue
    both_pass = ((pass_by_site[gt_col] == True) & (pass_by_site[other_col] == True)).sum()
    gt_only = ((pass_by_site[gt_col] == True) & (pass_by_site[other_col] == False)).sum()
    other_only = ((pass_by_site[gt_col] == False) & (pass_by_site[other_col] == True)).sum()
    print(f"{other_col}: both pass={both_pass}, ground-truth-only={gt_only}, {other_col}-only={other_only}")

In [ ]:
flagged_fits

# XX. Individual selected side IDs

In [ ]:
flagged_fits[
    (flagged_fits["passed"] == True)
    & (flagged_fits["variant"] == "Polarity-corrected + baseline estimate")
].columns

In [ ]:
cohort_meta[cohort_meta['site_id'] == SITE_ID]

## XX. Match original analysis

In [ ]:
from bms_sa_review.shared.aws_config import aq
from bms_sa_review.shared.ciccada_config import SAI

site_id_list_sql = ", ".join(str(int(s)) for s in sample_site_ids)
YEAR = 2024
MONTH = 9

######## IMPORTANT #########
# bms_sa_review/shared/ciccada_config.py defines TABLES["conformance_voltvar"] = "conformance_voltvar_v2_flex_included" 
# a different, newer physical table with extra methodology columns 
# need to query the literal string directly. DO NOT use TABLES["conformance_voltvar"] for this, or silently hit the different v2 table instead.

query = f"""
SELECT
    site_id,
    SUM(total_count)                      AS total_count,
    SUM(nonconformance_voltvar_red_count) AS nonconformance_red_count
FROM conformance_voltvar
WHERE site_id IN ({site_id_list_sql})
  AND year = {YEAR} AND month = {MONTH}
GROUP BY site_id
"""
site_conformance_sep24 = aq(query, database=SAI)
site_conformance_sep24["nonconformance_red_frac"] = (
    site_conformance_sep24["nonconformance_red_count"] / site_conformance_sep24["total_count"]
)
site_conformance_sep24["conformant_red"] = site_conformance_sep24["nonconformance_red_frac"] <= 0.10

matched = set(int(s) for s in site_conformance_sep24["site_id"])
missing = set(int(s) for s in sample_site_ids) - matched
print(f"Matched {len(matched)} / {len(sample_site_ids)} sampled sites in Sept 2024.")
print(f"Conformant (red): {site_conformance_sep24['conformant_red'].sum()} / {len(site_conformance_sep24)}")
print(f"No Sept-2024 rows at all: {sorted(missing)}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell: Side-by-side comparison -- slope analysis pass/fail vs the original
# SA analysis (Athena, conformance_voltvar) Sept-2024 label, restricted to
# the sites present in the slope analysis.
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

site_variant_passed = flagged_fits.pivot_table(
    index="site_id", columns="variant", values="passed", aggfunc="first"
)
slope_variant_cols = list(site_variant_passed.columns)  # the 4 DEFAULT_VVAR_VARIANTS names

# LEFT merge, anchored on the slope-analysis sites (65) -- any of the
# original SA analysis's extra sites (e.g. the 68 - 65 = 3 not in this
# sample) are simply absent from the join, not carried in as extra rows.
comparison = site_variant_passed.reset_index().merge(
    site_conformance_sep24[["site_id", "nonconformance_red_frac", "conformant_red"]],
    on="site_id", how="left",
)
comparison = comparison.rename(columns={
    "conformant_red": "Original SA analysis",
    "nonconformance_red_frac": "original_sa_nonconformance_red_frac",
})

original_sa_col = "Original SA analysis"
all_cols = slope_variant_cols + [original_sa_col]

# How many of the slope-analysis variants disagree with the original SA
# analysis label, per site (NaN -- no fit for that variant/site, e.g.
# excluded earlier for n_window<20, or no Sept-2024 row in the original SA
# table -- doesn't count as agreement OR disagreement, just "not compared").
comparison["n_compared"] = comparison[slope_variant_cols].notna().sum(axis=1)
comparison["n_disagree_with_original_sa"] = comparison.apply(
    lambda row: sum(
        (row[v] != row[original_sa_col])
        for v in slope_variant_cols
        if pd.notna(row[v]) and pd.notna(row[original_sa_col])
    ),
    axis=1,
)

comparison = comparison.sort_values(
    ["n_disagree_with_original_sa", "site_id"], ascending=[False, True]
).reset_index(drop=True)

print(f"{comparison['site_id'].nunique()} slope-analysis sites "
      f"({comparison[original_sa_col].notna().sum()} of them also found in the original SA "
      f"analysis's Sept-2024 table).")
display(comparison[["site_id"] + all_cols + ["original_sa_nonconformance_red_frac", "n_disagree_with_original_sa"]])

# --- Per-variant agreement rate with the original SA analysis ------------- #
agreement_rows = []
for v in slope_variant_cols:
    sub = comparison.dropna(subset=[v, original_sa_col])
    agreement_rows.append({
        "variant": v,
        "n_compared": len(sub),
        "pct_agree_with_original_sa": 100 * (sub[v] == sub[original_sa_col]).mean() if len(sub) else np.nan,
    })
agreement_summary = pd.DataFrame(agreement_rows)
print("\nAgreement with the original SA analysis label, per variant:")
display(agreement_summary)

In [ ]:
# --- Heatmap: site x (selected slope-analysis variants + original SA), pass/fail/missing -- #
heatmap_variant_cols = [
#    "Sign-flip only",
    "Polarity-corrected + baseline estimate",
    "Ground truth (ami_raw_phaseseparate)",
]
heatmap_cols = heatmap_variant_cols + [original_sa_col]

heat = comparison.set_index("site_id")[heatmap_cols]

code = np.full(heat.shape, -1.0)  # default: -1 = missing
for j, col in enumerate(heat.columns):
    vals = heat[col]
    present = vals.notna()
    code[present.to_numpy(), j] = vals[present].astype(bool).astype(float).to_numpy()

cmap = ListedColormap(["#e5e5e5", "#b91c1c", "#1b7f4c"])   # missing, fail, pass
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

fig, ax = plt.subplots(figsize=(1.2 * len(heatmap_cols) + 1.3, 0.16 * len(heat) + 1.2), dpi=130)
ax.imshow(code, cmap=cmap, norm=norm, aspect="auto")

for i in range(code.shape[0]):
    for j in range(code.shape[1]):
        symbol = {1: "\u2713", 0: "\u2717", -1: "\u2013"}[code[i, j]]
        ax.text(j, i, symbol, ha="center", va="center", fontsize=6, color="white" if code[i, j] != -1 else "#555555")

ax.set_xticks(range(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols, rotation=30, ha="right", fontsize=7)
ax.set_yticks(range(len(heat)))
ax.set_yticklabels(heat.index, fontsize=5)
ax.set_xticks(np.arange(-0.5, len(heatmap_cols), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(heat), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.2)
ax.tick_params(which="minor", length=0)
ax.set_title(
    f"Volt-VAr pass/fail vs original SA analysis "
    f"({MONTH} {YEAR}, \u00b1{TOLERANCE_PCT}% tolerance)",
    fontsize=9, fontweight="bold",
)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color="#1b7f4c", label="Pass / conformant"),
    plt.Rectangle((0, 0), 1, 1, color="#b91c1c", label="Fail / non-conformant"),
    plt.Rectangle((0, 0), 1, 1, color="#e5e5e5", label="No data / not compared"),
]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.04 - 0.002 * len(heat)),
          ncol=3, fontsize=7, frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# comparison.to_csv("voltvar_slope_analysis_vs_original_sa.csv", index=False)

In [ ]:
print(comparison.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrix: "Polarity-corrected + baseline estimate" vs original SA analysis
# ---------------------------------------------------------------------------
predicted_col = next(c for c in slope_variant_cols if "baseline" in c.lower())
reference_col = original_sa_col

cm_df = comparison.dropna(subset=[predicted_col, reference_col])[[predicted_col, reference_col]].copy()
cm_df[predicted_col] = cm_df[predicted_col].map({True: "Pass", False: "Fail"})
cm_df[reference_col] = cm_df[reference_col].map({True: "Pass", False: "Fail"})

confusion = pd.crosstab(
    cm_df[predicted_col], cm_df[reference_col],
    rownames=[predicted_col], colnames=[reference_col],
).reindex(index=["Pass", "Fail"], columns=["Pass", "Fail"], fill_value=0)

print(f"Confusion matrix ({len(cm_df)} sites with both labels present):")
display(confusion)

accuracy = (cm_df[predicted_col] == cm_df[reference_col]).mean()
print(f"Agreement: {accuracy:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 3), dpi=130)
ax.imshow(confusion.to_numpy(), cmap=ListedColormap(["#eef2f7"]), vmin=0, vmax=1)

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(confusion.iloc[i, j]), ha="center", va="center", fontsize=14, fontweight="bold")

ax.set_xticks([0, 1]); ax.set_xticklabels(confusion.columns, fontsize=9)
ax.set_yticks([0, 1]); ax.set_yticklabels(confusion.index, fontsize=9)
ax.set_xlabel(reference_col, fontsize=9)
ax.set_ylabel(predicted_col, fontsize=9)
ax.set_xticks([0.5], minor=True); ax.set_yticks([0.5], minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", length=0)
ax.set_title(f"Agreement: {accuracy:.1%} (n={len(cm_df)})", fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
site_breakdown = aq(f"""
    SELECT day_night,
           SUM(total_count) AS total_count,
           SUM(q_adverse_count) AS q_adverse_count,
           SUM(q_inactive_count) AS q_inactive_count,
           SUM(q_minor_deviation_count) AS q_minor_deviation_count,
           SUM(q_major_deficit_count) AS q_major_deficit_count,
           SUM(q_major_surplus_count) AS q_major_surplus_count,
           SUM(nonconformance_voltvar_red_count) AS red_count
    FROM conformance_voltvar
    WHERE site_id = 1440865990 AND year = 2024 AND month = 9
    GROUP BY day_night
""", database=SAI)
display(site_breakdown)

## 13. 

In [ ]:
variant_specs = [
    (axes[0], pv_scope_classified_flipped, rating_kw_flipped, "Q_kvar", "P_kw", "Sign-flip only"),
    (axes[1], pv_scope_classified, rating_kw, "Q_kvar_pv_est", "P_kw_pv_est", "Polarity-corrected + baseline estimate"),
    (axes[2], pv_scope_classified_true, rating_kw_true, "Q_kvar", "P_kw", "Ground truth (ami_raw_phaseseparate)"),
]
for ax, df, rating, q_col, p_col, label in variant_specs:
    window, binned, slope, intercept, fs = ConformancePlot._fit_assessable_window(
        df, rating, q_col, power_col=p_col)
    print(f"{label:<45} n={fs['n_window']:5d}  slope={slope:+.3f} %/V  R²={fs['r_squared']:.2f}")

In [ ]:
cohort_vvar_fits_signed = ConformancePlot.fetch_cohort_assessable_vvar_slopes(
    con, site_circuit_map,
    scope_where=scope_where,
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    metadata_table="ami_site_metadata_residential",
    variants=ConformancePlot.DEFAULT_VVAR_VARIANTS[1:],   # drop "Polarity-corrected only"
    signed_gate=True,
)
plotted_fits_signed = ConformancePlot.plot_cohort_assessable_vvar_slopes(
    cohort_vvar_fits_signed, min_n_window=20,
    variant_order=[v["name"] for v in ConformancePlot.DEFAULT_VVAR_VARIANTS[1:]])

# 13. Signed net-export capability gate

The existing `capability_assessable` gate is `abs(P) >= 20% rating`. On net-metered
variants, household **import** also passes it, which admits intervals where the PV
inverter produces ~nothing (the flat Q~0 cluster). The signed gate requires
**net export >= 20% rating** instead -- with no on-site storage, net export >= X
implies PV >= X, so it can't admit a non-generating interval. Applied post-hoc
to the same classified frames, so ground truth is unchanged (sanity check below).

In [ ]:
# ---------------------------------------------------------------------------
# 12.1 Example site: fetch each variant ONCE; the signed gate is then applied
# post-hoc to these same frames (no re-query needed for the second gate).
# ---------------------------------------------------------------------------
GATE_SITE_ID = 1084068901
GATE_LOAD_CIRCUIT_ID = 626819   # ac_load_net -- all net-metered variants
GATE_PV_CIRCUIT_ID = 626820     # pv_site_net -- ground truth only

gate_site_frames = {}
for variant in ConformancePlot.SIGNED_GATE_VVAR_VARIANTS:
    circuit_id = GATE_PV_CIRCUIT_ID if variant["circuit_col"] == "pv_circuit_id" else GATE_LOAD_CIRCUIT_ID
    classified_df, rating = ConformancePlot.fetch_circuit_scope_conformance(
        con, GATE_SITE_ID, circuit_id,
        scope_where=scope_where,
        rating_column="ac_capacity_kw",
        capability_profile="review_corrected",
        source_table=variant["source_table"],
        metadata_table="ami_site_metadata_residential",
        use_baseline_correction=variant["use_baseline_correction"],
    )
    gate_site_frames[variant["name"]] = (variant, classified_df, rating)

# Sanity check: export_sign must make midday P POSITIVE for every variant --
# otherwise the signed gate would keep import and drop export.
sign_check = pd.DataFrame({
    name: ConformancePlot.check_export_sign(df, v["power_col"], v["export_sign"])
    for name, (v, df, _) in gate_site_frames.items()
}).T
display(sign_check)
assert (sign_check["median_signed_p_kw"] > 0).all(), \
    "export_sign is backwards for at least one variant -- check _EXPORT_GATE_SPEC"

# When are the dropped points? (expect night / evening import, not midday)
v, df, rating = gate_site_frames["Sign-flip only"]
gated = ConformancePlot.apply_signed_export_gate(df, rating, v["power_col"], export_sign=v["export_sign"])
in_window = (gated["V"] >= ConformancePlot._VVAR["V3"]) & (gated["V"] < ConformancePlot._VW["V1"])
dropped = gated.loc[in_window & gated["dropped_by_signed_gate"]]
print(f"\nSign-flip only: {len(dropped):,} of {int((in_window & gated['capability_assessable_abs']).sum()):,} "
      f"assessable-window points dropped by the signed gate. By AEST hour:")
print(((dropped["t_stamp"].dt.hour + 10) % 24).value_counts().sort_index().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 12.2 Example site: kept vs dropped points, and the OLS line under each gate.
# "Polarity-corrected only" is left out (its Q is still load-convention, same
# reason the 3x3 day grid above leaves it out) -- add it back if wanted.
# ---------------------------------------------------------------------------
gate_panels = [
    dict(label=name, classified_df=df, rating_kw=rating, q_col=v["q_col"],
         power_col=v["power_col"], export_sign=v["export_sign"])
    for name, (v, df, rating) in gate_site_frames.items()
    if name != "Polarity-corrected only"
]

fig_gate_site, gate_site_results = ConformancePlot.plot_site_gate_comparison(
    gate_panels,
    daytime_hours=None,          # same as the cohort fits in Section 11
    suptitle=f"Site {GATE_SITE_ID}: |P| gate vs signed net-export gate — {scope_label}",
)
display(gate_site_results.round(3))

In [ ]:
# ---------------------------------------------------------------------------
# 12.3 Cohort: every (site, variant) under BOTH gates, one fetch each -- same
# runtime as Section 11's fetch_cohort_assessable_vvar_slopes cell.
# ---------------------------------------------------------------------------
cohort_gate_fits = ConformancePlot.fetch_cohort_vvar_slopes_gate_comparison(
    con, site_circuit_map,
    scope_where=scope_where,
    rating_column="ac_capacity_kw",
    capability_profile="review_corrected",
    metadata_table="ami_site_metadata_residential",
    daytime_hours=None,
)

display(cohort_gate_fits.groupby(["variant", "gate"])["status"]
        .apply(lambda s: s.str.split(":").str[0].value_counts()).unstack(fill_value=0))

# Ground truth must be unchanged by the signed gate (PV-only P is never negative).
gt = cohort_gate_fits.loc[(cohort_gate_fits["variant"] == ConformancePlot.GROUND_TRUTH_VARIANT_NAME)
                          & (cohort_gate_fits["status"] == "ok")]
gt_pivot = gt.pivot_table(index="site_id", columns="gate", values="n_window")
n_changed = int((gt_pivot[ConformancePlot.GATE_ABS] != gt_pivot[ConformancePlot.GATE_SIGNED]).sum())
print(f"\nGround-truth sites whose window changed under the signed gate: {n_changed} (expect 0)")

In [ ]:
# ---------------------------------------------------------------------------
# 12.4 Summary by (variant, gate): precision/recall of the assessable window
# vs ground truth's, and |slope error| vs ground truth. Same sites under both
# gates (paired_only=True).
# ---------------------------------------------------------------------------
gate_summary = ConformancePlot.summarise_gate_comparison(
    cohort_gate_fits, min_n_window=20, precision_threshold=0.95, paired_only=True,
)
display(gate_summary.round(3))

In [ ]:
# ---------------------------------------------------------------------------
# 12.5 Per-site slope error vs ground truth, |P| gate -> signed gate.
# ylim clips a few extreme sites (their count is printed in each title) --
# set ylim=None to see everything.
# ---------------------------------------------------------------------------
gate_slope_err_paired = ConformancePlot.plot_cohort_gate_slope_error(
    cohort_gate_fits, min_n_window=20,
    variant_order=["Sign-flip only", "Polarity-corrected + baseline estimate"],
    ylim=(-3, 3),
)

In [ ]:
# ---------------------------------------------------------------------------
# 12.6 Signed-gate fits through the SAME downstream helpers as Section 11
# (same columns as fetch_cohort_assessable_vvar_slopes, plus extras), and
# pass rates under each gate at the same TOLERANCE_PCT.
# ---------------------------------------------------------------------------
fits_signed = cohort_gate_fits.loc[cohort_gate_fits["gate"] == ConformancePlot.GATE_SIGNED].copy()
plotted_fits_signed = ConformancePlot.plot_cohort_assessable_vvar_slopes(fits_signed, min_n_window=20)

pass_rates = (
    pd.concat([
        ConformancePlot.add_slope_pass_fail(
            cohort_gate_fits.loc[(cohort_gate_fits["gate"] == g)
                                 & (cohort_gate_fits["status"] == "ok")
                                 & (cohort_gate_fits["n_window"] >= 20)],
            TOLERANCE_PCT,
        )
        for g in ConformancePlot.GATES
    ])
    .groupby(["variant", "gate"])["passed"].mean().mul(100).round(1)
    .unstack("gate")
)
print(f"% of sites passing the ±{TOLERANCE_PCT}% slope test, by gate:")
display(pass_rates)

In [ ]:
# ---------------------------------------------------------------------------
# 12.7 Sites where the signed gate still disagrees with ground truth -- either
# the window still contains non-assessable points (precision < 0.95) or the
# slope is still far off. These are the candidates for steps 2-3 (robust fit /
# offset handling) and for a site-level flag.
# ---------------------------------------------------------------------------
still_off = cohort_gate_fits.loc[
    (cohort_gate_fits["gate"] == ConformancePlot.GATE_SIGNED)
    & (cohort_gate_fits["variant"] != ConformancePlot.GROUND_TRUTH_VARIANT_NAME)
    & (cohort_gate_fits["status"] == "ok")
    & ((cohort_gate_fits["precision_vs_gt"] < 0.95) | (cohort_gate_fits["slope_err_vs_gt"].abs() > 1.0))
].sort_values("slope_err_vs_gt", key=lambda s: s.abs(), ascending=False)

display(still_off[["site_id", "variant", "n_window", "n_dropped_by_gate", "precision_vs_gt",
                   "recall_vs_gt", "slope", "gt_slope", "slope_err_vs_gt", "r_squared"]].round(3))